# Image → 3D on Colab (Stable Fast 3D, free T4)

Turns a **single image** into a clean, UV-unwrapped, **textured** 3D mesh using
[Stable Fast 3D (SF3D)](https://github.com/Stability-AI/stable-fast-3d) on a free Colab T4 —
**no HuggingFace ZeroGPU quota limits.**

Why SF3D for Roblox UGC: it's a *direct mesh* model (not gaussian-splatting), so it emits
**one coherent mesh** instead of the fragmented splat geometry + attached backdrop planes that
TRELLIS produces. It exposes the marketplace-prep controls we want: **triangle/quad remesh**,
a **vertex-count cap**, and a **2048 texture**.

SF3D needs **numpy<2**, but Colab ships numpy 2 — and you can't swap numpy in a live kernel.
So cell 3 builds an **isolated venv** with SF3D's own numpy<2 stack and runs SF3D as a
subprocess. Colab's kernel is never touched, so there's **no restart and no ABI clash**.

### How to run
1. **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. **Runtime → Run all.** Upload your image when cell 4 prompts. A `.glb` downloads at the end.

First run takes ~4–6 min (venv + torch + first-time CUDA op build). After that, generations are seconds.

## 1. Check the GPU (must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Accept the license + log in to HuggingFace

SF3D's weights are **gated**. One-time steps:
1. Open <https://huggingface.co/stabilityai/stable-fast-3d> and click **Agree / Access repository**.
2. Create a **read** token at <https://huggingface.co/settings/tokens>.
3. Run the cell below and paste the token (it is not saved to the notebook).

In [ ]:
from huggingface_hub import login
import getpass
login(token=getpass.getpass('HF token (read scope): '))
print('logged in')

## 3. Install SF3D (in an isolated venv)

Clones the repo and builds an isolated `venv` with SF3D's numpy<2 stack + its own torch,
then compiles the `texture_baker` / `uv_unwrapper` CUDA ops against it. This is the slow
part (~4–6 min first time). It ends by printing `venv OK | numpy 1.26.x | cuda True` —
if you see that, the environment is good and you can move on. **No restart needed.**

In [ ]:
import os
%cd /content
![ -d stable-fast-3d ] || git clone https://github.com/Stability-AI/stable-fast-3d.git

# WHY a venv: Colab's base env is numpy 2 (cudf/RAPIDS etc.), but SF3D's stack
# needs numpy<2, and you can't swap numpy in a live kernel. So build an ISOLATED
# venv and run SF3D as a subprocess — no ABI clash, no restart.
#
# NOTE: `python -m venv` fails on this Colab image (ensurepip is broken ->
# "No module named pip"), so we use `virtualenv`, which bundles pip directly.
!pip install -q virtualenv
!virtualenv -q -p python3 /content/sf3dvenv
PY = "/content/sf3dvenv/bin/python"
!{PY} -m pip install -q -U pip "setuptools==69.5.1" wheel ninja
# Torch CUDA 12.1 wheels (run on the T4 / sm_75).
!{PY} -m pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

%cd /content/stable-fast-3d
!sed -i 's/gpytoolbox==0.2.0/gpytoolbox==0.3.3/' requirements.txt
os.environ['CUDA_HOME'] = '/usr/local/cuda'
# Build the local CUDA ops against the VENV's torch (sm_75), no build isolation.
!CUDA_HOME=/usr/local/cuda TORCH_CUDA_ARCH_LIST=7.5 {PY} -m pip install -q --no-build-isolation ./texture_baker/ ./uv_unwrapper/
!sed -i '/texture_baker/d;/uv_unwrapper/d' requirements.txt
!{PY} -m pip install -q -r requirements.txt
!{PY} -m pip install -q -U "pymatting>=1.1.12"

# Sanity check INSIDE the venv: this MUST print "venv OK | numpy 1.26.x | cuda True".
!{PY} -c "import torch, sf3d.system, rembg, numpy; print('venv OK | numpy', numpy.__version__, '| cuda', torch.cuda.is_available())"

## 4. Upload your image

A clean subject on a plain background works best (SF3D removes the background automatically).
Run the cell, pick your file. To use a path instead, set `IMAGE_PATH` directly.

In [ ]:
from google.colab import files
import shutil, os
up = files.upload()
IMAGE_PATH = '/content/input' + os.path.splitext(next(iter(up)))[1]
shutil.move(next(iter(up)), IMAGE_PATH)
print('using', IMAGE_PATH)

## 5. Generate the mesh

Runs SF3D **inside the venv** (subprocess) in fp16 autocast — that's what keeps numpy<2
isolated and fits the T4's VRAM. It reads your HF token from the shared cache (cell 2),
so no re-login.

Tunables (top of the cell):
- `REMESH` = `triangle` (Roblox-friendly) / `quad` / `none`.
- `VERTEX_COUNT` = `6000` to land near the rigid-accessory budget (4,000 tris); `-1` = full.
- `TEXTURE_RES` = `1024` (T4-safe); `2048` = Roblox cap (more VRAM).

In [ ]:
import os, glob, subprocess
PY = "/content/sf3dvenv/bin/python"

REMESH = "triangle"     # triangle | quad | none
VERTEX_COUNT = 6000     # cap near the 4,000-tri budget; -1 = full
TEXTURE_RES = 1024      # T4-safe; 2048 = Roblox cap (more VRAM)

# Resolve the image uploaded in cell 4 (fall back to any /content/input*).
try:
    IMAGE_PATH
except NameError:
    _i = sorted(glob.glob('/content/input*'))
    assert _i, "No image found — run the Upload cell (4) first."
    IMAGE_PATH = _i[0]
print("input:", IMAGE_PATH)

runner = f'''
import sys, os, traceback
# run_sf3d.py lives in /content, so add the repo to the import path for `import sf3d`.
sys.path.insert(0, "/content/stable-fast-3d")
os.chdir("/content/stable-fast-3d")
try:
    from PIL import Image
    import torch, rembg, sf3d.utils as u
    from sf3d.system import SF3D
    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
    ip = sys.argv[1]
    m = SF3D.from_pretrained("stabilityai/stable-fast-3d",
                             config_name="config.yaml",
                             weight_name="model.safetensors").to("cuda").eval()
    img = Image.open(ip).convert("RGBA")
    img = u.remove_background(img, rembg.new_session())   # session from rembg directly
    img = u.resize_foreground(img, 0.85)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        mesh, _ = m.run_image(img, bake_resolution={TEXTURE_RES},
                              remesh="{REMESH}", vertex_count={VERTEX_COUNT})
    os.makedirs("/content/output", exist_ok=True)
    mesh.export("/content/output/mesh.glb", include_normals=True)
    print("DONE tris", len(mesh.faces))
except Exception:
    traceback.print_exc()
    sys.exit(1)
'''
with open("/content/run_sf3d.py", "w") as f:
    f.write(runner)

# Capture the venv subprocess output and surface it directly (so the real error
# is impossible to miss, even if only this cell's exception is copied).
ret = subprocess.run([PY, "/content/run_sf3d.py", IMAGE_PATH], capture_output=True, text=True)
print(ret.stdout)
if ret.stderr:
    print("----- venv stderr -----\n" + ret.stderr)
GLB = "/content/output/mesh.glb"
if ret.returncode != 0 or not os.path.exists(GLB):
    tail = "\n".join((ret.stdout + "\n" + ret.stderr).strip().splitlines()[-40:])
    raise RuntimeError("❌ SF3D generation failed — real error:\n" + tail)
print("mesh:", GLB)

## 6. Quick mesh stats + download

In [ ]:
!pip install -q trimesh
import glob, trimesh
# Use GLB from cell 5; fall back to the latest .glb under /content/output so this
# never NameErrors if cell 5's variable didn't carry over.
try:
    GLB
except NameError:
    _g = sorted(glob.glob('/content/output/**/*.glb', recursive=True))
    assert _g, "No GLB found — run the Generate cell (5) first."
    GLB = _g[-1]
m = trimesh.load(GLB, force='mesh')
print(f'tris: {len(m.faces):,}   verts: {len(m.vertices):,}   watertight: {m.is_watertight}')
print(f'bounds (units): {m.extents.round(3)}')
from google.colab import files
files.download(GLB)

## 7. Back in the repo

Drop the downloaded `.glb` into `runs/` and continue the pipeline:

```bash
# optional safety net (SF3D is already clean, so this should be a no-op):
roblox-ugc clean runs/shark/mesh.glb --out runs/shark/clean.glb

# import to Blender, decimate to the category tri budget, center, rescale:
roblox-ugc prep runs/shark/clean.glb --out runs/shark/prepped.fbx --decimate 4000 --center
roblox-ugc inspect runs/shark/prepped.fbx --out runs/shark/report.json
roblox-ugc validate runs/shark/report.json --target accessory --category Hat
```

Tip: pass `--target_vertex_count 6000` (cell 5) to be *born* near the rigid-accessory
4,000-tri cap, so the decimate step is light.

### Higher fidelity?
If you want richer geometry/texture and don't mind a slower run, **Hunyuan3D-2** also fits a T4
(<https://github.com/Tencent-Hunyuan/Hunyuan3D-2>) — octree mesh, also clean (no splat artifacts),
with controllable polygon count. SF3D is the fast/clean default; Hunyuan3D-2 is the quality step-up.

### ⚠️ Licensing (read before selling)
SF3D ships under the **Stability AI Community License**: free for research and for commercial use
**only if your annual revenue is ≤ US $1M** (above that needs a paid Stability Enterprise License).
The license does **not** explicitly address third-party **marketplace resale** (e.g. Roblox UGC) —
check the current `LICENSE.md` before submitting generated meshes for sale.

> Note: a **TPU** (e.g. v5e) can't run these models — they use custom CUDA kernels. Use the **T4 GPU** runtime.